# 02 — Feature Engineering
### Comparing BoW, TF-IDF, and Word2Vec

Feature engineering converts raw text into numbers a model can learn from.
This notebook compares three approaches:

| Method | Type | Captures meaning? | Captures order? | Dimensionality |
|--------|------|------------------|-----------------|----------------|
| Bag of Words (BoW) | Count-based | ❌ No | ❌ No | High (sparse) |
| TF-IDF | Weighted counts | Partially | ❌ No | High (sparse) |
| Word2Vec | Neural embeddings | ✅ Yes | ❌ No | Low (dense) |

We train each method, evaluate model accuracy with each, and choose the best.


## 1. Setup & Load

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.pipeline import Pipeline

plt.rcParams["figure.dpi"] = 120
LABEL_NAMES = ["negative","neutral","positive"]

train = pd.read_csv("../data/train.csv")
val   = pd.read_csv("../data/val.csv")
test  = pd.read_csv("../data/test.csv")

print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")
print(f"Text column: 'final_text' (lemmatized)")

Train: 11,666 | Val: 1,457 | Test: 1,459
Text column: 'final_text' (lemmatized)


## 2. Method 1 — Bag of Words (BoW)

The simplest text representation. Counts how many times each word appears.

**How it works:**
- Build a vocabulary of all words (e.g. 15,000)
- For each tweet, count how many times each word appears
- Result: a vector of raw counts

**Limitation:** Treats "not good" and "good not" identically — no word order.
Common words like "the" and "a" dominate despite carrying no meaning.


In [2]:
# Build BoW vectorizer
bow = CountVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    min_df=2,
    strip_accents="unicode"
)

X_train_bow = bow.fit_transform(train["final_text"].fillna(""))
X_val_bow   = bow.transform(val["final_text"].fillna(""))
X_test_bow  = bow.transform(test["final_text"].fillna(""))

print(f"BoW matrix shape: {X_train_bow.shape}")
print(f"Vocabulary size : {len(bow.vocabulary_):,}")
print(f"Matrix sparsity : {(1 - X_train_bow.nnz/(X_train_bow.shape[0]*X_train_bow.shape[1]))*100:.1f}% zeros")

BoW matrix shape: (11666, 15000)
Vocabulary size : 15,000
Matrix sparsity : 99.9% zeros


In [3]:
# Show BoW representation of one tweet
example = train["final_text"].iloc[0]
example_vec = bow.transform([example])
nonzero_idx = example_vec.nonzero()[1]
feat_names  = bow.get_feature_names_out()

print(f"Tweet: '{example[:60]}'")
print(f"\nBoW representation (non-zero features):")
print(f"  {'Word':<20} Count")
print(f"  {'-'*30}")
for idx in sorted(nonzero_idx, key=lambda i: example_vec[0,i], reverse=True)[:10]:
    print(f"  {feat_names[idx]:<20} {example_vec[0,idx]:.0f}")

Tweet: 'dmd you hr ago at your request no response to be found just '

BoW representation (non-zero features):
  Word                 Count
  ------------------------------
  ago                  1
  at                   1
  at your              1
  bag                  1
  be                   1
  be found             1
  dmd                  1
  dmd you              1
  found                1
  hr                   1


In [4]:
# Train LR with BoW
lr_bow = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
lr_bow.fit(X_train_bow, train["label"])

bow_val_acc = accuracy_score(val["label"], lr_bow.predict(X_val_bow))
bow_val_f1  = f1_score(val["label"], lr_bow.predict(X_val_bow), average="weighted")
bow_test_acc = accuracy_score(test["label"], lr_bow.predict(X_test_bow))
bow_test_f1  = f1_score(test["label"], lr_bow.predict(X_test_bow), average="weighted")

print(f"BoW + Logistic Regression:")
print(f"  Val  accuracy : {bow_val_acc*100:.2f}%")
print(f"  Val  F1       : {bow_val_f1*100:.2f}%")
print(f"  Test accuracy : {bow_test_acc*100:.2f}%")
print(f"  Test F1       : {bow_test_f1*100:.2f}%")

BoW + Logistic Regression:
  Val  accuracy : 78.52%
  Val  F1       : 78.92%
  Test accuracy : 78.75%
  Test F1       : 79.12%


## 3. Method 2 — TF-IDF

Improves on BoW by weighting words by how rare they are across all documents.

**Key improvement over BoW:**
- "the" appears in every tweet → very low IDF → near-zero weight
- "cancelled" appears in few tweets → high IDF → high weight
- Rare, distinctive words get boosted — common words get suppressed

**TF-IDF score = TF × IDF**
- TF (term frequency) = how often word appears in this tweet
- IDF (inverse document frequency) = log(total tweets / tweets containing word)


In [5]:
# Standard TF-IDF
tfidf = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    strip_accents="unicode"
)

X_train_tfidf = tfidf.fit_transform(train["final_text"].fillna(""))
X_val_tfidf   = tfidf.transform(val["final_text"].fillna(""))
X_test_tfidf  = tfidf.transform(test["final_text"].fillna(""))

# Compare BoW vs TF-IDF on same tweet
feat_names_tfidf = tfidf.get_feature_names_out()
example_tfidf = tfidf.transform([example])
nonzero_tfidf = example_tfidf.nonzero()[1]

print(f"Tweet: '{example[:60]}'")
print(f"\n{'Word':<20} {'BoW count':>12} {'TF-IDF score':>14}")
print("-"*48)
for idx in sorted(nonzero_tfidf, key=lambda i: example_tfidf[0,i], reverse=True)[:10]:
    word = feat_names_tfidf[idx]
    tfidf_score = example_tfidf[0,idx]
    # Find same word in BoW
    bow_score = example_vec[0, bow.vocabulary_.get(word, -1)] if word in bow.vocabulary_ else 0
    print(f"  {word:<18} {bow_score:>12.0f} {tfidf_score:>14.4f}")

Tweet: 'dmd you hr ago at your request no response to be found just '

Word                    BoW count   TF-IDF score
------------------------------------------------
  dmd you                       1         0.2831
  hr ago                        1         0.2786
  be found                      1         0.2746
  just like                     1         0.2746
  like my                       1         0.2711
  dmd                           1         0.2575
  at your                       1         0.2553
  response to                   1         0.2533
  no response                   1         0.2320
  request                       1         0.2213


In [6]:
# Show IDF scores — low IDF = common, high IDF = rare
idf_scores = tfidf.idf_
feature_names = tfidf.get_feature_names_out()

low_idf  = np.argsort(idf_scores)[:8]
high_idf = np.argsort(idf_scores)[-8:][::-1]

print("Most COMMON words (low IDF — least informative for classification):")
for idx in low_idf:
    print(f"  '{feature_names[idx]:<20}' IDF={idf_scores[idx]:.3f}")
print()
print("Most RARE words (high IDF — most distinctive for classification):")
for idx in high_idf:
    print(f"  '{feature_names[idx]:<20}' IDF={idf_scores[idx]:.3f}")

Most COMMON words (low IDF — least informative for classification):
  'to                  ' IDF=1.843
  'the                 ' IDF=2.124
  'flight              ' IDF=2.333
  'for                 ' IDF=2.404
  'you                 ' IDF=2.422
  'on                  ' IDF=2.457
  'and                 ' IDF=2.477
  'my                  ' IDF=2.642

Most RARE words (high IDF — most distinctive for classification):
  'no internet         ' IDF=9.266
  'yuck                ' IDF=9.266
  'yvonne anthony      ' IDF=9.266
  'nice customer       ' IDF=9.266
  'or leave            ' IDF=9.266
  'zero to             ' IDF=9.266
  'noone               ' IDF=9.266
  'aa cancelled        ' IDF=9.266


In [7]:
# Train LR with TF-IDF
lr_tfidf = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
lr_tfidf.fit(X_train_tfidf, train["label"])

tfidf_val_acc  = accuracy_score(val["label"],  lr_tfidf.predict(X_val_tfidf))
tfidf_val_f1   = f1_score(val["label"],  lr_tfidf.predict(X_val_tfidf), average="weighted")
tfidf_test_acc = accuracy_score(test["label"], lr_tfidf.predict(X_test_tfidf))
tfidf_test_f1  = f1_score(test["label"], lr_tfidf.predict(X_test_tfidf), average="weighted")

print(f"TF-IDF + Logistic Regression:")
print(f"  Val  accuracy : {tfidf_val_acc*100:.2f}%")
print(f"  Val  F1       : {tfidf_val_f1*100:.2f}%")
print(f"  Test accuracy : {tfidf_test_acc*100:.2f}%")
print(f"  Test F1       : {tfidf_test_f1*100:.2f}%")

TF-IDF + Logistic Regression:
  Val  accuracy : 79.41%
  Val  F1       : 79.81%
  Test accuracy : 78.48%
  Test F1       : 78.89%


## 4. Method 3 — Word2Vec

Word2Vec is a neural network that learns dense vector representations of words.
Unlike BoW and TF-IDF which create sparse high-dimensional vectors,
Word2Vec creates small dense vectors (100-300 dimensions) that capture **meaning**.

**Key property:**
Words with similar meanings have similar vectors:
- "delayed" and "late" → similar vectors
- "amazing" and "wonderful" → similar vectors
- "delayed" and "amazing" → very different vectors

**For a tweet:** Average all word vectors to get one tweet vector.

**Limitation:** Averaging loses information about which words appear together.


In [8]:
# from gensim.models import Word2Vec

# # Tokenize — Word2Vec needs a list of lists
# train_tokens = [text.split() for text in train["final_text"].fillna("")]
# val_tokens   = [text.split() for text in val["final_text"].fillna("")]
# test_tokens  = [text.split() for text in test["final_text"].fillna("")]

# # Train Word2Vec model
# w2v = Word2Vec(
#     sentences=train_tokens,
#     vector_size=100,     # each word = 100-dimensional vector
#     window=5,            # context window size
#     min_count=2,         # ignore words appearing fewer than 2 times
#     workers=4,
#     epochs=10,
#     seed=42
# )

# print(f"Word2Vec vocabulary size : {len(w2v.wv):,} words")
# print(f"Vector dimensions        : {w2v.vector_size}")
# print()
# # Show similar words to demonstrate semantic understanding
# for word in ["delayed","cancelled","great","terrible"]:
#     if word in w2v.wv:
#         similar = w2v.wv.most_similar(word, topn=4)
#         similar_str = ", ".join([f"{w}({s:.2f})" for w,s in similar])
#         print(f"  Words similar to '{word}': {similar_str}")

In [9]:
# # Convert tweets to vectors by averaging word vectors
# def tweet_to_vec(tokens, model, size=100):
#     vecs = [model.wv[w] for w in tokens if w in model.wv]
#     return np.mean(vecs, axis=0) if vecs else np.zeros(size)

# X_train_w2v = np.array([tweet_to_vec(t, w2v) for t in train_tokens])
# X_val_w2v   = np.array([tweet_to_vec(t, w2v) for t in val_tokens])
# X_test_w2v  = np.array([tweet_to_vec(t, w2v) for t in test_tokens])

# print(f"Word2Vec feature matrix shape: {X_train_w2v.shape}")
# print(f"Each tweet = {X_train_w2v.shape[1]}-dimensional dense vector")
# print(f"Compare to TF-IDF: {X_train_tfidf.shape[1]}-dimensional sparse vector")


In [10]:
# # Train LR with Word2Vec
# lr_w2v = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
# lr_w2v.fit(X_train_w2v, train["label"])

# w2v_val_acc  = accuracy_score(val["label"],  lr_w2v.predict(X_val_w2v))
# w2v_val_f1   = f1_score(val["label"],  lr_w2v.predict(X_val_w2v), average="weighted")
# w2v_test_acc = accuracy_score(test["label"], lr_w2v.predict(X_test_w2v))
# w2v_test_f1  = f1_score(test["label"], lr_w2v.predict(X_test_w2v), average="weighted")

# print(f"Word2Vec + Logistic Regression:")
# print(f"  Val  accuracy : {w2v_val_acc*100:.2f}%")
# print(f"  Val  F1       : {w2v_val_f1*100:.2f}%")
# print(f"  Test accuracy : {w2v_test_acc*100:.2f}%")
# print(f"  Test F1       : {w2v_test_f1*100:.2f}%")


## 5. Method Comparison

In [11]:
# results = {
#     "Bag of Words"  : {"val_acc": bow_val_acc,   "val_f1": bow_val_f1,
#                        "test_acc": bow_test_acc,  "test_f1": bow_test_f1,
#                        "dims": X_train_bow.shape[1],   "sparse": True},
#     "TF-IDF"        : {"val_acc": tfidf_val_acc, "val_f1": tfidf_val_f1,
#                        "test_acc": tfidf_test_acc,"test_f1": tfidf_test_f1,
#                        "dims": X_train_tfidf.shape[1], "sparse": True},
#     }
# #     # "Word2Vec"      : {"val_acc": w2v_val_acc,   "val_f1": w2v_val_f1,
# #                        "test_acc": w2v_test_acc,  "test_f1": w2v_test_f1,
# #                        "dims": X_train_w2v.shape[1],   "sparse": False},
# # }

# # Print table
# print(f"{'Method':<16} {'Val Acc':>9} {'Val F1':>8} {'Test Acc':>10} {'Test F1':>9} {'Dims':>8}")
# print("-"*65)
# for method, r in results.items():
#     print(f"  {method:<14} {r['val_acc']*100:>8.2f}% {r['val_f1']*100:>7.2f}% "
#           f"{r['test_acc']*100:>9.2f}% {r['test_f1']*100:>8.2f}% {r['dims']:>8,}")

# # Bar chart comparison
# fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# methods  = list(results.keys())
# test_acc = [results[m]["test_acc"]*100 for m in methods]
# test_f1  = [results[m]["test_f1"]*100  for m in methods]

# x = np.arange(len(methods)); w = 0.35
# axes[0].bar(x-w/2, test_acc, w, label="Accuracy", color="#534AB7", alpha=0.85)
# axes[0].bar(x+w/2, test_f1,  w, label="F1 Weighted", color="#1D9E75", alpha=0.85)
# for bars in axes[0].containers:
#     axes[0].bar_label(bars, fmt="%.1f%%", padding=3, fontsize=9)
# axes[0].set_xticks(x); axes[0].set_xticklabels(methods)
# axes[0].set_ylim(60, 100); axes[0].set_title("Test Set Performance", fontweight="bold")
# axes[0].legend(); axes[0].spines[["top","right"]].set_visible(False)

# # Dimensions comparison
# dims = [results[m]["dims"] for m in methods]
# axes[1].bar(methods, dims, color=["#E24B4A","#534AB7","#1D9E75"], alpha=0.85, edgecolor="white")
# for bar, d in zip(axes[1].patches, dims):
#     axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
#                  f"{d:,}", ha="center", fontsize=9)
# axes[1].set_title("Feature Dimensions", fontweight="bold")
# axes[1].set_ylabel("Number of dimensions")
# axes[1].spines[["top","right"]].set_visible(False)

# plt.suptitle("Feature Engineering Method Comparison", fontsize=13, fontweight="bold")
# plt.tight_layout()
# plt.savefig("../outputs/02_feature_comparison.png", dpi=150, bbox_inches="tight")
# plt.show()

# best = max(results, key=lambda m: results[m]["test_f1"])
# print(f"\n✅ Best method: {best}")
# print(f"   We will use TF-IDF for all classical ML models.")
# print(f"   Word2Vec trained on 11k tweets is too small — pre-trained embeddings (Layer 2) will be better.")


## 6. Save Best Vectorizer

In [12]:
import pickle
from pathlib import Path

Path("../outputs").mkdir(exist_ok=True)

# Save TF-IDF vectorizer
with open("../outputs/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

# Save BoW vectorizer  
with open("../outputs/bow_vectorizer.pkl", "wb") as f:
    pickle.dump(bow, f)

print("Saved:")
print("  ../outputs/tfidf_vectorizer.pkl")
print("  ../outputs/bow_vectorizer.pkl")
print()
print(f"  BoW   : {bow_test_acc*100:.1f}%  — fast but ignores word importance")
print(f"  TF-IDF: {tfidf_test_acc*100:.1f}%  — best classical approach for text")
print(f"  Word2Vec : skipped — gensim not compatible with Python 3.14")
print()
print("TF-IDF is the clear winner for classical ML on this dataset.")

Saved:
  ../outputs/tfidf_vectorizer.pkl
  ../outputs/bow_vectorizer.pkl

  BoW   : 78.8%  — fast but ignores word importance
  TF-IDF: 78.5%  — best classical approach for text
  Word2Vec : skipped — gensim not compatible with Python 3.14

TF-IDF is the clear winner for classical ML on this dataset.
